# Fairness processing methods

## Requirements

In [75]:
import numpy as np
import pandas as pd 

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split 

from aif360.datasets import GermanDataset
from aif360.sklearn.datasets import fetch_german
from aif360.sklearn.metrics import statistical_parity_difference
from aif360.sklearn.preprocessing import Reweighing, ReweighingMeta
from aif360.sklearn.inprocessing import (AdversarialDebiasing, ExponentiatedGradientReduction, 
                                         GridSearchReduction)
from aif360.sklearn.postprocessing import CalibratedEqualizedOdds, RejectOptionClassifier, PostProcessingMeta

import tensorflow as tf

from fairlearn.reductions import Moment # necessary for ExponentiatedGradientReduction

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

## Initial elements

### Data

In [76]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [77]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
# The sensitive variable/s must index the df in order to avoid problems with some aif360 objects
german_data_pd.index = german_data_pd[sens_variable_name] 
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [79]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [80]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [81]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

## Pre-processing

In [82]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
log_reg_estimator.fit(X_train, Y_train)
Y_test_hat = log_reg_estimator.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set

In [83]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.32539682539682535

In [84]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

### Reweighing

In [85]:
class ReweighingMetaEstimator(BaseEstimator, ClassifierMixin):
   
    def __init__(self, estimator, prot_attr):
        
        self.estimator = estimator
        self.prot_attr = prot_attr

    def fit(self, X, y, sample_weight=None):

        # X must be a Pandas DataFrame
        
        A = X[self.prot_attr]
        reweigher_ = Reweighing(prot_attr=A)
        self.meta_estimator = ReweighingMeta(estimator=self.estimator, reweigher=reweigher_)
        
        # Handle sample_weight if provided
        if sample_weight is not None:
            # Normalize the weights if necessary (ensure they sum to 1 or another form)
            sample_weight = sample_weight / sample_weight.sum()
            # Resample X and y based on sample_weight
            X_resampled, y_resampled = self._resample_with_weights(X, y, sample_weight)
            self.meta_estimator.fit(X_resampled, y_resampled)
        else:
            self.meta_estimator.fit(X, y)

        self.classes_ = self.meta_estimator.classes_

        return self
    
    def predict(self, X):

        return self.meta_estimator.predict(X)
    
    def predict_proba(self, X):

        return self.meta_estimator.predict_proba(X)
    
    def _resample_with_weights(self, X, y, sample_weight):
        """Resample dataset according to sample weights."""
        import numpy as np

        # Scale sample weights to integer values for resampling
        weights = np.round(sample_weight * len(sample_weight)).astype(int)

        # Resample X and y
        if isinstance(X, np.ndarray):
            X_resampled = np.repeat(X, weights, axis=0)
        elif isinstance(X, pd.DataFrame):
            X_resampled = pd.DataFrame(np.repeat(X.values, weights, axis=0), columns=X.columns)
        if isinstance(y, np.ndarray):
            y_resampled = np.repeat(y, weights, axis=0)
        elif isinstance(y, pd.Series):
            y_resampled = pd.Series(np.repeat(y.values, weights), name=y.name)

        return X_resampled, y_resampled

In [134]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)
reweighing_log_reg_estimator = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)
reweighing_log_reg_estimator.fit(X=X_train, y=Y_train)
Y_test_hat_reweighing = reweighing_log_reg_estimator.predict(X_test)

In [135]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat_reweighing, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

In [136]:
balanced_accuracy_score(y_pred=Y_test_hat_reweighing, y_true=Y_test)

0.6396825396825396

## In-processing

In [89]:
A_train = X_train[sens_variable_name] # sensitive variable / protected attribute in train set

### Adversarial Debiasing

In [90]:
class AdversarialDebiasingEstimator(BaseEstimator, ClassifierMixin):
    
    def __init__(self, prot_attr, scope_name, adversary_loss_weight, num_epochs, 
                 batch_size, classifier_num_hidden_units, debias, verbose, random_state):
        
        self.prot_attr = prot_attr
        self.scope_name = scope_name
        self.adversary_loss_weight = adversary_loss_weight
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.classifier_num_hidden_units = classifier_num_hidden_units
        self.debias = debias
        self.verbose = verbose
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        
        # Extract protected attribute
        A = X[self.prot_attr]

        # Initialize the original AdversarialDebiasing model
        self.estimator = AdversarialDebiasing(prot_attr=A,
                                              scope_name=self.scope_name,
                                              adversary_loss_weight=self.adversary_loss_weight,
                                              num_epochs=self.num_epochs,
                                              batch_size=self.batch_size,
                                              classifier_num_hidden_units=self.classifier_num_hidden_units,
                                              debias=self.debias,
                                              verbose=self.verbose,
                                              random_state=self.random_state)
        
        # Handle sample_weight if provided
        if sample_weight is not None:
            # Normalize the weights if necessary (ensure they sum to 1 or another form)
            sample_weight = sample_weight / sample_weight.sum()
            # Resample X and y based on sample_weight
            X_resampled, y_resampled = self._resample_with_weights(X, y, sample_weight)
            self.estimator.fit(X_resampled, y_resampled)
        else:
            self.estimator.fit(X, y)
        
        self.classes_ = self.estimator.classes_

        return self

    def predict(self, X):
        return self.estimator.predict(X)

    def predict_proba(self, X):
        return self.estimator.predict_proba(X)

    def _resample_with_weights(self, X, y, sample_weight):
        """Resample dataset according to sample weights."""
        import numpy as np

        # Scale sample weights to integer values for resampling
        weights = np.round(sample_weight * len(sample_weight)).astype(int)

        # Resample X and y
        if isinstance(X, np.ndarray):
            X_resampled = np.repeat(X, weights, axis=0)
        elif isinstance(X, pd.DataFrame):
            X_resampled = pd.DataFrame(np.repeat(X.values, weights, axis=0), columns=X.columns)
        if isinstance(y, np.ndarray):
            y_resampled = np.repeat(y, weights, axis=0)
        elif isinstance(y, pd.Series):
            y_resampled = pd.Series(np.repeat(y.values, weights), name=y.name)

        return X_resampled, y_resampled


In [91]:
# Set up TensorFlow session (required by AdversarialDebiasing)
tf_session = tf.compat.v1.Session
# disable_eager_execution is required as well by TensorFlow
tf.compat.v1.disable_eager_execution()

In [92]:
adversarial_debiasing = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

In [93]:
adversarial_debiasing.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

AdversarialDebiasingEstimator(adversary_loss_weight=0.1, batch_size=128,
                              classifier_num_hidden_units=200, debias=True,
                              num_epochs=50, prot_attr='age', random_state=123,
                              scope_name='classifier', verbose=True)

In [94]:
Y_test_hat = adversarial_debiasing.predict(X_test)
Y_test_hat

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [95]:
adversarial_debiasing.predict_proba(X_test)

array([[4.35757606e-02, 9.56424239e-01],
       [3.26964053e-01, 6.73035947e-01],
       [1.65164924e-02, 9.83483508e-01],
       [3.68154000e-04, 9.99631846e-01],
       [8.42234403e-02, 9.15776560e-01],
       [2.01420627e-02, 9.79857937e-01],
       [1.17699089e-02, 9.88230091e-01],
       [8.73835584e-02, 9.12616442e-01],
       [2.43701875e-05, 9.99975630e-01],
       [1.01378968e-06, 9.99998986e-01],
       [5.59886449e-03, 9.94401136e-01],
       [1.55193219e-01, 8.44806781e-01],
       [3.10029329e-01, 6.89970671e-01],
       [1.61254638e-01, 8.38745362e-01],
       [2.15432678e-02, 9.78456732e-01],
       [1.47831134e-05, 9.99985217e-01],
       [2.71572412e-01, 7.28427588e-01],
       [2.11611720e-03, 9.97883883e-01],
       [2.68423368e-03, 9.97315766e-01],
       [1.78996229e-01, 8.21003771e-01],
       [1.19439669e-01, 8.80560331e-01],
       [3.68624113e-07, 9.99999631e-01],
       [2.09801817e-03, 9.97901982e-01],
       [7.40108719e-03, 9.92598913e-01],
       [6.391627

In [96]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [97]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

### Exponentiated Gradient Reduction

In [98]:
class ExponentiatedGradientReductionEstimator(BaseEstimator, ClassifierMixin):
    
    def __init__(self, prot_attr, estimator, constraints, eps, 
                 max_iter, nu, eta0, run_linprog_step, drop_prot_attr):
        
        self.prot_attr = prot_attr
        self.estimator = estimator
        self.constraints = constraints
        self.eps = eps
        self.max_iter = max_iter
        self.nu = nu
        self.eta0 = eta0
        self.run_linprog_step = run_linprog_step
        self.drop_prot_attr = drop_prot_attr

    def fit(self, X, y, sample_weight=None):
        
        # Initialize the original AdversarialDebiasing model
        self.meta_estimator = ExponentiatedGradientReduction(prot_attr=self.prot_attr, estimator=self.estimator, 
                                                             constraints=self.constraints, eps=self.eps, max_iter=self.max_iter, 
                                                             nu=self.nu, eta0=self.eta0, run_linprog_step=self.run_linprog_step, 
                                                             drop_prot_attr=self.drop_prot_attr)
        
        # Handle sample_weight if provided
        if sample_weight is not None:
            # Normalize the weights if necessary (ensure they sum to 1 or another form)
            sample_weight = sample_weight / sample_weight.sum()
            # Resample X and y based on sample_weight
            X_resampled, y_resampled = self._resample_with_weights(X, y, sample_weight)
            self.meta_estimator.fit(X_resampled, y_resampled)
        else:
            self.meta_estimator.fit(X, y)
        
        self.classes_ = self.meta_estimator.classes_

        return self

    def predict(self, X):
        return self.meta_estimator.predict(X)

    def predict_proba(self, X):
        return self.meta_estimator.predict_proba(X)

    def _resample_with_weights(self, X, y, sample_weight):
        """Resample dataset according to sample weights."""
        import numpy as np

        # Scale sample weights to integer values for resampling
        weights = np.round(sample_weight * len(sample_weight)).astype(int)

        # Resample X and y
        if isinstance(X, np.ndarray):
            X_resampled = np.repeat(X, weights, axis=0)
        elif isinstance(X, pd.DataFrame):
            X_resampled = pd.DataFrame(np.repeat(X.values, weights, axis=0), columns=X.columns)
        if isinstance(y, np.ndarray):
            y_resampled = np.repeat(y, weights, axis=0)
        elif isinstance(y, pd.Series):
            y_resampled = pd.Series(np.repeat(y.values, weights), name=y.name)

        return X_resampled, y_resampled


In [99]:
eg_red = ExponentiatedGradientReduction(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                                        constraints='ErrorRateParity', 
                                        eps=0.01, max_iter=50, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

eg_red.fit(X=X_train, y=Y_train)

# Some possible values for constraints: 'DemographicParity', 'EqualizedOdds', 'TruePositiveRateParity', 'ErrorRateParity'

ExponentiatedGradientReduction(constraints='ErrorRateParity',
                               drop_prot_attr=False,
                               estimator=LogisticRegression(random_state=123,
                                                            solver='liblinear'),
                               prot_attr='age')

In [100]:
Y_test_hat = eg_red.predict(X_test)

In [101]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6142857142857143

In [102]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.259920634920635

### GridSearchReduction

In [103]:
class GridSearchReductionEstimator(BaseEstimator, ClassifierMixin):
    
    def __init__(self, prot_attr, estimator, constraints, constraint_weight, grid_size, 
                       grid_limit, grid, drop_prot_attr, loss, min_val, max_val):
        
        self.prot_attr = prot_attr
        self.estimator = estimator
        self.constraints = constraints
        self.constraint_weight = constraint_weight
        self.grid_size = grid_size
        self.grid_limit = grid_limit
        self.grid = grid
        self.loss = loss
        self.min_val = min_val
        self.max_val = max_val
        self.drop_prot_attr = drop_prot_attr

    def fit(self, X, y, sample_weight=None):
        
        # Initialize the original AdversarialDebiasing model
        self.meta_estimator = GridSearchReduction(prot_attr=self.prot_attr, estimator=self.estimator, 
                                                  constraints=self.constraints, constraint_weight=self.constraint_weight, 
                                                  grid_size=self.grid_size, grid_limit=self.grid_limit, grid=self.grid, 
                                                  drop_prot_attr=self.drop_prot_attr, loss=self.loss, 
                                                  min_val=self.min_val, max_val=self.max_val)
        
        # Handle sample_weight if provided
        if sample_weight is not None:
            # Normalize the weights if necessary (ensure they sum to 1 or another form)
            sample_weight = sample_weight / sample_weight.sum()
            # Resample X and y based on sample_weight
            X_resampled, y_resampled = self._resample_with_weights(X, y, sample_weight)
            self.meta_estimator.fit(X_resampled, y_resampled)
        else:
            self.meta_estimator.fit(X, y)
        
        self.classes_ = self.meta_estimator.classes_

        return self

    def predict(self, X):
        return self.meta_estimator.predict(X)

    def predict_proba(self, X):
        return self.meta_estimator.predict_proba(X)

    def _resample_with_weights(self, X, y, sample_weight):
        """Resample dataset according to sample weights."""
        import numpy as np

        # Scale sample weights to integer values for resampling
        weights = np.round(sample_weight * len(sample_weight)).astype(int)

        # Resample X and y
        if isinstance(X, np.ndarray):
            X_resampled = np.repeat(X, weights, axis=0)
        elif isinstance(X, pd.DataFrame):
            X_resampled = pd.DataFrame(np.repeat(X.values, weights, axis=0), columns=X.columns)
        if isinstance(y, np.ndarray):
            y_resampled = np.repeat(y, weights, axis=0)
        elif isinstance(y, pd.Series):
            y_resampled = pd.Series(np.repeat(y.values, weights), name=y.name)

        return X_resampled, y_resampled


In [104]:
gs_red = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

In [105]:
gs_red.fit(X_train, Y_train)

GridSearchReductionEstimator(constraint_weight=0.5,
                             constraints='ErrorRateParity',
                             drop_prot_attr=False,
                             estimator=LogisticRegression(random_state=123,
                                                          solver='liblinear'),
                             grid=None, grid_limit=2.0, grid_size=10,
                             loss='ZeroOne', max_val=None, min_val=None,
                             prot_attr='age')

In [106]:
Y_test_hat = gs_red.predict(X_test)

In [107]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6428571428571428

In [108]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.3769841269841269

## Post-processing

### PostProcessingMetaEstimator

In [119]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

ppm = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

ppm.fit(X_train, Y_train)

PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                solver='liblinear'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [120]:
Y_test_hat = ppm.predict(X_test)

In [121]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5222222222222223

In [122]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.5

In [114]:
roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

ppm = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=roc,
                         prefit=False, val_size=0.25)

ppm.fit(X_train, Y_train)

PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                solver='liblinear'),
                   postprocessor=RejectOptionClassifier(prot_attr='age'))

In [116]:
Y_test_hat = ppm.predict(X_test)

In [117]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6444444444444445

In [118]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.05555555555555558

## Combining processing methods

### Pre + In 

- in pro (ExponentiatedGradientReduction) with pre_pro (ReweighingMetaEstimator) as estimator --> ERROR

In [126]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

In [127]:
in_pro = ExponentiatedGradientReduction(prot_attr=sens_variable_name, estimator=pre_pro, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

in_pro.fit(X=X_train, y=Y_train)
Y_test_hat = in_pro.predict(X_test)

ValueError: Length mismatch: Expected 834 rows, received array of length 850

---

- pre_pro (ReweighingMetaEstimator) with in pro (AdversarialDebiasingEstimator) as estimator --> Same results as using the in pro (AdversarialDebiasingEstimator) alone --> Doesn't work as expected

In [116]:
in_pro = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

ReweighingMetaEstimator(estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                                batch_size=128,
                                                                classifier_num_hidden_units=200,
                                                                debias=True,
                                                                num_epochs=50,
                                                                prot_attr='age',
                                                                random_state=123,
                                                                scope_name='classifier',
                                                                verbose=True),
                        prot_attr='age')

In [117]:
Y_test_hat = pre_pro.predict(X_test)

In [118]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [119]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

---

- pre_pro (ReweighingMetaEstimator) with in pro (ExponentiatedGradientReduction) as estimator --> WORKS! --> different results than when using only the in-pro or only the pre-pro

In [ ]:
in_pro = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                                                 constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

In [100]:
Y_test_hat = pre_pro.predict(X_test)

In [101]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6444444444444445

In [102]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.23412698412698407

In [104]:
in_pro.fit(X_train, Y_train)
Y_test_hat = in_pro.predict(X_test)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

In [105]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6412698412698412

In [106]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.251984126984127

In [107]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)
Y_test_hat = pre_pro.predict(X_test)

In [108]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6396825396825396

In [109]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.17658730158730163

---

- pre_pro (ReweighingMetaEstimator) with in pro (GridSearchReduction) as estimator --> Same results as using the in pro (GridSearchReductionEstimator) alone --> Doesn't work as expected

In [152]:
in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

pre_pro = ReweighingMetaEstimator(estimator=in_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

ReweighingMetaEstimator(estimator=GridSearchReductionEstimator(constraint_weight=0.5,
                                                               constraints='ErrorRateParity',
                                                               drop_prot_attr=False,
                                                               estimator=LogisticRegression(random_state=123,
                                                                                            solver='liblinear'),
                                                               grid=None,
                                                               grid_limit=2.0,
                                                               grid_size=10,
                                                               loss='ZeroOne',
                                                               max_val=None,
                                                               min_val=None,
                                                               prot_attr='age'),
                        prot_attr='age')

In [153]:
Y_test_hat = pre_pro.predict(X_test)

In [154]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6428571428571428

In [155]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.3769841269841269

In [125]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=pre_pro, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

in_pro.fit(X_train, Y_train)

ValueError: Length mismatch: Expected 1100 rows, received array of length 850

### In + In 

- in pro (ExponentiatedGradientReduction) with in pro (AdversarialDebiasingEstimator) as estimator --> Doestn work as expected --> Same results as when using only in pro (AdversarialDebiasingEstimator)

In [72]:
adversarial_debiasing = AdversarialDebiasingEstimator(   
                                                        prot_attr=sens_variable_name,               
                                                        scope_name='classifier',            
                                                        adversary_loss_weight=0.1,           
                                                        num_epochs=50,                      
                                                        batch_size=128,                     
                                                        classifier_num_hidden_units=200,     
                                                        debias=True,                        
                                                        verbose=True,                     
                                                        random_state=123                   
                                                        )

In [73]:
eg_red = ExponentiatedGradientReduction(prot_attr=sens_variable_name, estimator=adversarial_debiasing, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

eg_red.fit(X=X_train, y=Y_train)

c:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\.venv\Lib\site-packages\fairlearn\reductions\_moments\utility_parity.py:214: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self.pos_basis[i]["+", e, g] = 1
c:\Users\fscielzo

epoch   0; iter:    0; batch classifier loss: 76.1198; batch adversarial loss: 0.6261
epoch   1; iter:    0; batch classifier loss: 63.8298; batch adversarial loss: 0.5519
epoch   2; iter:    0; batch classifier loss: 60.7581; batch adversarial loss: 0.5742
epoch   3; iter:    0; batch classifier loss: 30.8277; batch adversarial loss: 0.5774
epoch   4; iter:    0; batch classifier loss: 43.2861; batch adversarial loss: 0.5405
epoch   5; iter:    0; batch classifier loss: 34.1161; batch adversarial loss: 0.5866
epoch   6; iter:    0; batch classifier loss: 36.6167; batch adversarial loss: 0.6283
epoch   7; iter:    0; batch classifier loss: 41.4618; batch adversarial loss: 0.5844
epoch   8; iter:    0; batch classifier loss: 35.4636; batch adversarial loss: 0.6084
epoch   9; iter:    0; batch classifier loss: 39.0727; batch adversarial loss: 0.5631
epoch  10; iter:    0; batch classifier loss: 39.0982; batch adversarial loss: 0.5938
epoch  11; iter:    0; batch classifier loss: 43.7595;

ExponentiatedGradientReduction(constraints='EqualizedOdds',
                               drop_prot_attr=False,
                               estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                                       batch_size=128,
                                                                       classifier_num_hidden_units=200,
                                                                       debias=True,
                                                                       num_epochs=50,
                                                                       prot_attr='age',
                                                                       random_state=123,
                                                                       scope_name='classifier',
                                                                       verbose=True),
                               max_iter=10, prot_attr='age')

In [74]:
Y_test_hat = eg_red.predict(X_test)

In [75]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.49523809523809526

In [76]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.007936507936507908

### Pre + Post


In [129]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

In [130]:
pre_pro = ReweighingMetaEstimator(estimator=post_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)

ReweighingMetaEstimator(estimator=PostProcessingMeta(estimator=LogisticRegression(random_state=123,
                                                                                  solver='liblinear'),
                                                     postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                                                           random_state=123)),
                        prot_attr='age')

In [131]:
Y_test_hat = pre_pro.predict(X_test)

In [132]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5857142857142857

In [133]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.3214285714285714

---

In [143]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=roc,
                         prefit=False, val_size=0.25)

In [144]:
pre_pro = ReweighingMetaEstimator(estimator=post_pro, prot_attr=sens_variable_name)
pre_pro.fit(X_train, Y_train)
Y_test_hat = pre_pro.predict(X_test)

In [145]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6571428571428571

In [146]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.045634920634920584

---

In [138]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=pre_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

PostProcessingMeta(estimator=ReweighingMetaEstimator(estimator=LogisticRegression(random_state=123,
                                                                                  solver='liblinear'),
                                                     prot_attr='age'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [140]:
Y_test_hat = in_pro.predict(X_test)

In [141]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5253968253968254

In [142]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.33333333333333337

---

In [147]:
pre_pro = ReweighingMetaEstimator(estimator=log_reg_estimator, prot_attr=sens_variable_name)

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 

post_pro = PostProcessingMeta(estimator=pre_pro, postprocessor=roc,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)
Y_test_hat = post_pro.predict(X_test)

In [148]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6444444444444445

In [149]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.04365079365079361

### In + Post

In [159]:
in_pro = ExponentiatedGradientReduction(prot_attr=sens_variable_name, estimator=log_reg_estimator, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

PostProcessingMeta(estimator=ExponentiatedGradientReduction(constraints='EqualizedOdds',
                                                            drop_prot_attr=False,
                                                            estimator=LogisticRegression(random_state=123,
                                                                                         solver='liblinear'),
                                                            max_iter=10,
                                                            prot_attr='age'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [160]:
Y_test_hat = post_pro.predict(X_test)

In [ ]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

In [ ]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

---

In [163]:
ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                             prefit=False, val_size=0.25)

in_pro = ExponentiatedGradientReduction(prot_attr=sens_variable_name, estimator=post_pro, constraints='EqualizedOdds', 
                                        eps=0.01, max_iter=10, nu=None, eta0=2.0, 
                                        run_linprog_step=True, drop_prot_attr=False)

in_pro.fit(X_train, Y_train)

KeyError: "None of ['age'] are in the columns"

In [ ]:
Y_test_hat = in_pro.predict(X_test)

---

In [151]:
in_pro = AdversarialDebiasingEstimator(prot_attr=sens_variable_name,               
                                                      scope_name='classifier',            
                                                      adversary_loss_weight=0.1,           
                                                      num_epochs=50,                      
                                                      batch_size=128,                     
                                                      classifier_num_hidden_units=200,     
                                                      debias=True,                        
                                                      verbose=True,                     
                                                      random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

epoch   0; iter:    0; batch classifier loss: 72.7854; batch adversarial loss: 0.6374
epoch   1; iter:    0; batch classifier loss: 59.8815; batch adversarial loss: 0.6464
epoch   2; iter:    0; batch classifier loss: 44.1747; batch adversarial loss: 0.6032
epoch   3; iter:    0; batch classifier loss: 52.1853; batch adversarial loss: 0.6016
epoch   4; iter:    0; batch classifier loss: 52.4041; batch adversarial loss: 0.5648
epoch   5; iter:    0; batch classifier loss: 39.1910; batch adversarial loss: 0.5503
epoch   6; iter:    0; batch classifier loss: 44.7757; batch adversarial loss: 0.5920
epoch   7; iter:    0; batch classifier loss: 47.0720; batch adversarial loss: 0.5435
epoch   8; iter:    0; batch classifier loss: 44.7651; batch adversarial loss: 0.6199
epoch   9; iter:    0; batch classifier loss: 38.3134; batch adversarial loss: 0.5768
epoch  10; iter:    0; batch classifier loss: 35.5115; batch adversarial loss: 0.6024
epoch  11; iter:    0; batch classifier loss: 29.9044;

PostProcessingMeta(estimator=AdversarialDebiasingEstimator(adversary_loss_weight=0.1,
                                                           batch_size=128,
                                                           classifier_num_hidden_units=200,
                                                           debias=True,
                                                           num_epochs=50,
                                                           prot_attr='age',
                                                           random_state=123,
                                                           scope_name='classifier',
                                                           verbose=True),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [152]:
Y_test_hat = post_pro.predict(X_test)

In [153]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.5

In [154]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

0.0

----

In [155]:
in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=log_reg_estimator, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=in_pro, postprocessor=ceo,
                             prefit=False, val_size=0.25)

post_pro.fit(X_train, Y_train)

PostProcessingMeta(estimator=GridSearchReductionEstimator(constraint_weight=0.5,
                                                          constraints='ErrorRateParity',
                                                          drop_prot_attr=False,
                                                          estimator=LogisticRegression(random_state=123,
                                                                                       solver='liblinear'),
                                                          grid=None,
                                                          grid_limit=2.0,
                                                          grid_size=10,
                                                          loss='ZeroOne',
                                                          max_val=None,
                                                          min_val=None,
                                                          prot_attr='age'),
                   postprocessor=CalibratedEqualizedOdds(prot_attr='age',
                                                         random_state=123))

In [156]:
Y_test_hat = post_pro.predict(X_test)

In [157]:
balanced_accuracy_score(y_pred=Y_test_hat, y_true=Y_test)

0.6190476190476191

In [158]:
statistical_parity_difference(y_true=Y_test,
                              y_pred=Y_test_hat, 
                              prot_attr=A_test,
                              priv_group=sens_privileged_groups[0][sens_variable_name], 
                              pos_label=response_favorable_label)

-0.36706349206349204

---

In [166]:
ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

post_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                             prefit=False, val_size=0.25)

in_pro = GridSearchReductionEstimator(prot_attr=sens_variable_name, estimator=post_pro, 
                             constraints='ErrorRateParity', constraint_weight=0.5, grid_size=10, 
                             grid_limit=2.0, grid=None, drop_prot_attr=False, loss='ZeroOne', 
                             min_val=None, max_val=None)

in_pro.fit(X_train, Y_train)

KeyError: "None of ['age'] are in the columns"

### Post + Post

In [167]:
log_reg_estimator = LogisticRegression(solver='liblinear', random_state=123)

ceo = CalibratedEqualizedOdds(sens_variable_name, cost_constraint='weighted', random_state=123) 

roc = RejectOptionClassifier(prot_attr=sens_variable_name, threshold=0.5, margin=0.1) 


post1_pro = PostProcessingMeta(estimator=log_reg_estimator, postprocessor=ceo,
                         prefit=False, val_size=0.25)

post2_pro = PostProcessingMeta(estimator=post1_pro, postprocessor=roc,
                         prefit=False, val_size=0.25)

post2_pro.fit(X_train, Y_train)

TypeError: `estimator` (type: <class 'aif360.sklearn.postprocessing.PostProcessingMeta'>) does not implement method `predict_proba()`.